# RoadExtension V1 — 통합 PM10 시각화

**구성**
- 대기 PM10 (V5-base ST-GNN)
- 도로 재비산먼지 기여분 (RF)
- 통합 PM10 = 대기 + 도로 기여

**날짜/시간 설정은 2번 셀에서**

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ── 한글 폰트 ──────────────────────────────────────────────────────────────
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 경로 ──────────────────────────────────────────────────────────────────
_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, 'config.py')):
    V1_DIR = _cwd
elif os.path.isfile(os.path.join(_cwd, 'RoadExtension_V1', 'config.py')):
    V1_DIR = os.path.join(_cwd, 'RoadExtension_V1')
else:
    V1_DIR = '/workspace/ST-GNN Modeling/RoadExtension_V1'

CKPT   = os.path.join(V1_DIR, 'checkpoints')
GDIR   = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본'

# ── 데이터 로드 ────────────────────────────────────────────────────────────
ambient  = np.load(os.path.join(CKPT, '../../../HiddenExtension_V5/checkpoints/V5-base/grid_pm10_test.npy'))
road_con = np.load(os.path.join(CKPT, 'road_contribution_test.npy'))
total_pm = np.load(os.path.join(CKPT, 'total_pm10_test.npy'))

grid_csv = pd.read_csv(os.path.join(GDIR, '격자_250m_4326.csv'))
lon      = grid_csv['lon'].values
lat      = grid_csv['lat'].values

ts_all   = np.load(os.path.join(GDIR, 'timestamps_all.npy'))
tidx     = np.load(os.path.join(GDIR, 'time_idx_test.npy'))
ts_test  = pd.to_datetime(ts_all[tidx[12:]])

with open(os.path.join(CKPT, 'combine_result.json')) as f:
    combine_info = json.load(f)

BLUE_CMAP = LinearSegmentedColormap.from_list('pm_blue',
    ['#f7fbff','#deebf7','#9ecae1','#4292c6','#2171b5','#084594','#03003d'], N=256)
RED_CMAP  = LinearSegmentedColormap.from_list('road_red',
    ['#fff5f0','#fee0d2','#fc9272','#de2d26','#a50f15'], N=256)

print(f'데이터 로드 완료')
print(f'테스트 기간: {ts_test[0].date()} ~ {ts_test[-1].date()}')
print(f'대기 PM  평균: {ambient.mean():.2f} μg/m³')
print(f'도로 기여 평균: {road_con.mean():.2f} μg/m³ ({combine_info["road_fraction_pct"]:.1f}%)')
print(f'통합 PM  평균: {total_pm.mean():.2f} μg/m³')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  날짜와 시간을 여기서 설정하세요
TARGET_DATE = '2025-08-15'
TARGET_HOUR = 9
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

t_idx    = np.abs(ts_test - pd.Timestamp(f'{TARGET_DATE} {TARGET_HOUR:02d}:00:00')).argmin()
ts_label = ts_test[t_idx].strftime('%Y년 %m월 %d일 %H시')

am_t    = ambient[t_idx]
rd_t    = road_con[t_idx]
tot_t   = total_pm[t_idx]

print(f'선택 시점: {ts_label}')
print(f'  대기 PM  : mean={am_t.mean():.2f}  max={am_t.max():.2f}')
print(f'  도로 기여: mean={rd_t.mean():.2f}  max={rd_t.max():.2f}')
print(f'  통합 PM  : mean={tot_t.mean():.2f}  max={tot_t.max():.2f}')

In [ ]:
# ── 3-panel 비교 지도: 대기 / 도로기여 / 통합 ────────────────────────────
def scatter_map(ax, lon, lat, vals, cmap, vmin, vmax, title, cbar_label):
    sc = ax.scatter(lon, lat, c=vals, cmap=cmap, vmin=vmin, vmax=vmax,
                    s=3.0, linewidths=0, alpha=0.92, rasterized=True)
    cb = plt.colorbar(sc, ax=ax, fraction=0.028, pad=0.02, shrink=0.75)
    cb.set_label(cbar_label, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('경도', fontsize=9); ax.set_ylabel('위도', fontsize=9)
    ax.set_aspect('equal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

vmax_pm   = max(np.percentile(am_t, 98), np.percentile(tot_t, 98))
vmax_road = np.percentile(rd_t, 98)

fig, axes = plt.subplots(1, 3, figsize=(22, 8), facecolor='white')
for ax in axes: ax.set_facecolor('white')

scatter_map(axes[0], lon, lat, am_t,  BLUE_CMAP, 0, vmax_pm,
            '대기 PM10\n(V5-base ST-GNN)', 'PM10 (μg/m³)')
scatter_map(axes[1], lon, lat, rd_t,  RED_CMAP,  0, vmax_road,
            '도로 재비산 기여분\n(RF 예측)', 'PM10 (μg/m³)')
scatter_map(axes[2], lon, lat, tot_t, BLUE_CMAP, 0, vmax_pm,
            '통합 PM10\n(대기 + 도로 기여)', 'PM10 (μg/m³)')

fig.suptitle(f'서울 PM10 통합 지도  |  {ts_label}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 도로 기여 비율 지도 + 히스토그램 ─────────────────────────────────────
road_pct_t = rd_t / (tot_t + 1e-8) * 100   # (G,) 각 격자 도로 기여 %

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='white')
for ax in axes: ax.set_facecolor('white')

# 도로 기여 비율 지도
pct_cmap = LinearSegmentedColormap.from_list('pct',
    ['#f0f0f0','#fdd49e','#fdbb84','#fc8d59','#e34a33','#b30000'], N=256)
scatter_map(axes[0], lon, lat, road_pct_t.clip(0, 50), pct_cmap, 0, 50,
            '도로 재비산 기여 비율', '기여 비율 (%)')

# 기여 비율 히스토그램
axes[1].hist(road_pct_t, bins=50, color='#e34a33', edgecolor='white', alpha=0.85)
axes[1].axvline(road_pct_t.mean(), color='black', linewidth=2,
                label='평균 {:.1f}%'.format(road_pct_t.mean()))
axes[1].set_xlabel('도로 기여 비율 (%)', fontsize=11)
axes[1].set_ylabel('격자 수', fontsize=11)
axes[1].set_title('격자별 도로 기여 비율 분포', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

fig.suptitle(f'도로 재비산먼지 기여 분석  |  {ts_label}', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 기간 평균 3-panel 지도 ──────────────────────────────────────────────
am_mean   = ambient.mean(axis=0)
rd_mean   = road_con.mean(axis=0)
tot_mean  = total_pm.mean(axis=0)

vmax_pm_m  = max(np.percentile(am_mean,  98), np.percentile(tot_mean, 98))
vmax_rd_m  = np.percentile(rd_mean, 98)

fig, axes = plt.subplots(1, 3, figsize=(22, 8), facecolor='white')
for ax in axes: ax.set_facecolor('white')

scatter_map(axes[0], lon, lat, am_mean,  BLUE_CMAP, 0, vmax_pm_m,
            '대기 PM10 기간 평균\n(V5-base)', 'PM10 (μg/m³)')
scatter_map(axes[1], lon, lat, rd_mean,  RED_CMAP,  0, vmax_rd_m,
            '도로 기여분 기간 평균\n(RF)', 'PM10 (μg/m³)')
scatter_map(axes[2], lon, lat, tot_mean, BLUE_CMAP, 0, vmax_pm_m,
            '통합 PM10 기간 평균', 'PM10 (μg/m³)')

period = '{} ~ {}'.format(ts_test[0].strftime('%Y.%m.%d'), ts_test[-1].strftime('%Y.%m.%d'))
fig.suptitle('서울 PM10 통합 기간 평균  |  {}'.format(period),
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('=== 전체 기간 통계 ===')
print('  대기 PM   mean: {:.2f} μg/m³'.format(am_mean.mean()))
print('  도로 기여 mean: {:.2f} μg/m³ ({:.1f}%)'.format(
    rd_mean.mean(), rd_mean.mean()/(tot_mean.mean()+1e-8)*100))
print('  통합 PM   mean: {:.2f} μg/m³'.format(tot_mean.mean()))